# Libraries

In [1]:
import pandas as pd
import os
from langchain.prompts import ChatPromptTemplate
from langchain.chat_models import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain.callbacks import get_openai_callback
from tqdm.auto import tqdm
from langchain.chat_models import ChatAnthropic
from langchain_google_genai import ChatGoogleGenerativeAI

In [2]:
%pip install --upgrade --quiet anthropic
%pip install --upgrade --quiet langchain-anthropic
%pip install --upgrade --quiet  langchain-google-genai pillow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 24.1
[notice] To update, run: C:\Users\user\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 24.1
[notice] To update, run: C:\Users\user\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 24.1
[notice] To update, run: C:\Users\user\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


# API Keys

In [6]:
os.environ["OPENAI_API_KEY"] = "hidden"
os.environ["ANTHROPIC_API_KEY"] = "hidden"
os.environ["GOOGLE_API_KEY"] = "hidden"

In [4]:
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")
GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")

# Docstring generation prompts for functions

In [7]:
system_prompt_func = """
You are a helpful assistant, assinged with generating docstrings to python functions. 
You have vast knowledge of python coding and libraries.
You know and follow the google docstring conventions.
"""

my_prompt_func = """
Provide a good docstring to the following code: {code}

Correct format example:

code = 
def multiply_and_sum(lst):
    res = 0
    for i, item in enumerate(lst):
        res += item * i 
    return res

solution =
    Iteratively compute the sum of all elements in a list of integers after multiplying each element by its index in the list.
    
    Args:
    lst (list): a list of integers.
    
    Returns:
    int: the sum of all elements in the list after preforming the multipication of the elements by their indices.
    
Return the docstring only (the string literal). 
Do not include the code or the triple quotes of the docstring.
Output should be like using '__doc__'.
Stick to this format.
"""

# Docstring generation prompts for classes

In [8]:
system_prompt_class = """
You are a helpful assistant, assinged with generating docstrings to python classes. 
You have vast knowledge of python coding and libraries.
You know and follow the google docstring conventions.
"""

my_prompt_class = """
Provide a good docstring to the following code: {code}

Correct format example:

code = 
class BankAccount:
    def __init__(self, owner, initial_balance=0.0):
        self.owner = owner
        self.balance = initial_balance
    def deposit(self, amount):
        if amount > 0:
            self.balance += amount
        else:
            raise ValueError("Deposit amount must be positive")
    def withdraw(self, amount):
        if amount > self.balance:
            raise ValueError("Insufficient funds")
        elif amount <= 0:
            raise ValueError("Withdrawal amount must be positive")
        else:
            self.balance -= amount
    def get_balance(self):
        return self.balance

solution =
    A class used to represent a bank account.

    Attributes:
    ----------
    owner : string 
        the name of the account owner.
    balance : float 
        the current balance of the account.

    Methods:
    -------
    __init__(owner, initial_balance=0.0)
        initializes the BankAccount with an owner and an optional initial balance.
    deposit(amount)
        deposits the specified amount into the account.
    withdraw(amount)
        withdraws the specified amount from the account if funds are available.
    get_balance()
        returns the current balance of the account.
    
Return the docstring only (the string literal). 
Do not include the code or the triple quotes of the docstring.
Output should be like using '__doc__'.
Stick to this format.
"""

# LLM Setup

In [9]:
prompt_func = ChatPromptTemplate.from_messages([
    ("system", system_prompt_func),
    ("user", my_prompt_func),
])

prompt_class = ChatPromptTemplate.from_messages([
    ("system", system_prompt_class),
    ("user", my_prompt_class),
])


# gemini doesn't have a good system prompt option built in, so we will concatenate the system and user prompts and use that as the prompt for gemini.

prompt_for_gemini_func = ChatPromptTemplate.from_template(
    system_prompt_func + my_prompt_func
)

prompt_for_gemini_class = ChatPromptTemplate.from_template(
    system_prompt_class + my_prompt_class
)

In [10]:
def generate_docstring_llm(code, docstring_col, chain_solution):
    llm_docstring = chain_solution.invoke({
        "code": code
    })
    return docstring_col, llm_docstring

## OPEN AI

In [11]:
llm_openai = ChatOpenAI(openai_api_key=OPENAI_API_KEY, model="gpt-3.5-turbo")

output_parser = StrOutputParser()

chain_func_gpt_turbo = prompt_func | llm_openai | output_parser
chain_class_gpt_turbo = prompt_class | llm_openai | output_parser

C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\langchain_core\_api\deprecation.py:119: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 0.2.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_openai import ChatOpenAI`.
  warn_deprecated(


Generating docstring using 'gpt-3.5 turbo' just for one function and class (before generating for all the data)

In [12]:
data = pd.read_csv("T5_Baseline_docstring_generation.csv")
function_2_code = data.loc[2, 'Function']
class_4_code = data.loc[43, 'Function']

In [18]:
generated_docstring_function_2_gpt_turbo = generate_docstring_llm(function_2_code, 'GPT-3.5 Turbo', chain_func_gpt_turbo)[1]
generated_docstring_class_4_gpt_turbo = generate_docstring_llm(class_4_code, 'GPT-3.5 Turbo', chain_class_gpt_turbo)[1]

print(f"Function 2 docstring: \n{generated_docstring_function_2_gpt_turbo}")
print(f"\nClass 4 docstring: \n{generated_docstring_class_4_gpt_turbo}")

Function 2 docstring: 

Calculate the sum of all elements in a nested list after converting all elements to floats and taking their absolute values.

    Args:
    lst (list): A nested list of integers or strings.

    Returns:
    float: The sum of all elements in the nested list after converting them to floats and taking their absolute values.

Class 4 docstring: 
A class representing a point in 2D space.

Attributes:
----------
x : int or float
    x-coordinate of the point.
y : int or float
    y-coordinate of the point.
r : float
    distance from the origin to the point.
theta : float
    angle in radians with respect to the positive x-axis.

Methods:
-------
__init__(x, y)
    Initializes the Point_2D with x and y coordinates.
__repr__()
    Returns a string representation of the point.
__eq__(other)
    Checks if two points are equal based on their coordinates.
__add__(other)
    Adds two points together and returns a new Point_2D object.
__sub__(other)
    Subtracts one point 

# ANTHROPIC

In [13]:
llm_anthropic = ChatAnthropic(anthropic_api_key=ANTHROPIC_API_KEY, model_name="claude-instant-1")

chain_func_claude = prompt_func | llm_anthropic | output_parser
chain_class_claude = prompt_class | llm_anthropic | output_parser

In [19]:
generated_docstring_function_2_claude = generate_docstring_llm(function_2_code, 'claude', chain_func_claude)[1]
generated_docstring_class_4_claude = generate_docstring_llm(class_4_code, 'claude', chain_class_claude)[1]

print(f"Function 2 docstring: \n{generated_docstring_function_2_claude}")
print(f"\nClass 4 docstring: \n{generated_docstring_class_4_claude}")

Function 2 docstring: 
 Sum the absolute value of nested lists or elements recursively.

This function recursively sums the absolute value of elements in a nested list. It checks the type of the first element and handles strings by converting to float and lists by recursively calling itself on the sublists and combining results. For other types like integers it simply takes the absolute value and recursively processes the rest of the list.

Class 4 docstring: 
 """A class representing a 2D point in cartesian coordinate space.

Attributes:
----------
x : float
   the x coordinate of the point
y : float
   the y coordinate of the point 
r : float
   the distance from the origin
theta : float
   the angle between the point and the positive x-axis

Methods:
--------
__repr__()
   returns a string representation of the point
__eq__() 
   checks for equality between two points
__add__()
   returns a new point that is the sum of this point and another 
__sub__()
   returns a new point that is

# GOOGLE

In [14]:
llm_google = ChatGoogleGenerativeAI(google_api_key=GOOGLE_API_KEY, model="gemini-1.0-pro")

chain_func_gemini = prompt_for_gemini_func | llm_google | output_parser
chain_class_gemini = prompt_for_gemini_class | llm_google | output_parser

In [17]:
generated_docstring_function_2_gemini = generate_docstring_llm(function_2_code, 'Gemini', chain_func_gemini)[1]
generated_docstring_class_4_gemini = generate_docstring_llm(class_4_code, 'Gemini', chain_class_gemini)[1]

print(f"Function 2 docstring: \n{generated_docstring_function_2_gemini}")
print(f"\nClass 4 docstring: \n{generated_docstring_class_4_gemini}")

Function 2 docstring: 
Iteratively compute the sum of all elements in a list after taking the absolute value of each element.
    
    Args:
    lst (list): a list of integers or lists.
    
    Returns:
    float: the sum of all elements in the list after taking the absolute value of each element.

Class 4 docstring: 
"""A 2D point class with attributes x, y, r and theta.

    Attributes:
    ----------
    x : float 
        the x-coordinate of the point.
    y : float 
        the y-coordinate of the point.
    r : float 
        the distance from the point to the origin.
    theta : float 
        the angle (in radians) between the positive x-axis and the line connecting the point to the origin.

    Methods:
    -------
    __init__(x, y)
        initializes the Point_2D with x and y coordinates.
    __repr__()
        returns a string representation of the point.
    __eq__(other)
        compares the point to another point and returns True if they are equal.
    __add__(other)
 

# Generate docstrings for all the data using all 3 models

In [16]:
# Split the data into functions and classes.
data_full = pd.read_csv("T5_Baseline_docstring_generation.csv")
data_funcs = data_full.iloc[:40]
data_classes = data_full.iloc[40:]

# List of models to generate docstrings with, sorted by usage price in case of bugs.
models_funcs = [('Gemini-1.0-pro', chain_func_gemini), ('GPT-3.5 Turbo', chain_func_gpt_turbo), ('Claude-instant-1', chain_func_claude)]
models_classes = [('Gemini-1.0-pro', chain_class_gemini), ('GPT-3.5 Turbo', chain_class_gpt_turbo), ('Claude-instant-1', chain_class_claude)]

tqdm.pandas()

def generate_docstrings_pipeline(data, models):
    for model_name, model_chain in models:
        new_docstring_col = model_name
        data[new_docstring_col] = None
        missing_values_mask = data[new_docstring_col].isna()
        print(f"\nGenerating docstrings by {new_docstring_col}...")
        data.loc[missing_values_mask, [new_docstring_col]] = (
            data.loc[missing_values_mask, 'Function'].progress_apply(
                lambda x: generate_docstring_llm(x, new_docstring_col, model_chain)[1]
            )
        )
        # Debugging: Print the generated docstrings
        print(data.loc[missing_values_mask, [new_docstring_col]])
    return data

In [20]:
# Run the pipeline on the functions data
data_funcs = generate_docstrings_pipeline(data_funcs, models_funcs)


Generating docstrings by Gemini-1.0-pro...


C:\Users\user\AppData\Local\Temp\ipykernel_29276\435095776.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data[new_docstring_col] = None


  0%|          | 0/40 [00:00<?, ?it/s]

Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 Resource has been exhausted (e.g. check quota)..
Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 4.0 seconds as it raised ResourceExhausted: 429 Resource has been exhausted (e.g. check quota)..
Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 8.0 seconds as it raised ResourceExhausted: 429 Resource has been exhausted (e.g. check quota)..
Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 16.0 seconds as it raised ResourceExhausted: 429 Resource has been exhausted (e.g. check quota)..


                                       Gemini-1.0-pro
0   Computes the sum of all even-indexed elements ...
1   Recursively compute the number of possible way...
2   Recursively compute the sum of a nested list, ...
3   Compute the number of ways to decompose a targ...
4   Computes the binomial coefficient of n and k.\...
5   Iteratively perform a DFS level order traversa...
6   Iteratively find a subset of integers in a lis...
7   Iteratively compute the edit distance between ...
8   Iteratively checks if there are directed cycle...
9   Recursively computes the value of the sequence...
10  Iteratively compute the difference between two...
11  Iteratively compute the longest increasing sub...
12  Iteratively compute the median of a list of in...
13  Finds the prime factors of a given integer n.\...
14  Finds the intersection between two graphs and ...
15  Find all subsets of a list that sum to a given...
16  Iteratively evaluate an expression that contai...
17  Iteratively search for a

...

  0%|          | 0/40 [00:00<?, ?it/s]

                                        GPT-3.5 Turbo
0   \n    Recursively calculate the sum of even el...
1   \nFind the number of ways to represent a given...
2   \n    Recursively calculates the sum of elemen...
3   \n    Recursively decomposes a target string b...
4   \nCalculate the number of ways to choose k ele...
5   \n    Perform a depth-first search traversal o...
6   \n    Find a subset of the input list whose el...
7   \nCompute the minimum string distance between ...
8   \n    Check if a given graph is a directed acy...
9   \n    Recursively calculate the final values o...
10  \n    Calculate the difference between multipl...
11  \nCalculate the length of the longest subseque...
12  \nFind the median of a list of numbers using t...
13  \n    Find the primary factors of a given inte...
14  \n    Find the intersection of nodes between t...
15  Iteratively find all subsets in a list whose e...
16  Iteratively compute the result of a mathematic...
17  \nCheck if any substring

C:\Users\user\AppData\Local\Temp\ipykernel_29276\435095776.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data[new_docstring_col] = None


  0%|          | 0/40 [00:00<?, ?it/s]

                                     Claude-instant-1
0    Iteratively sum the values in a list if they ...
1    """Find all possible ways to make change for ...
2    """Iteratively compute the absolute sum of ne...
3    """Iteratively decompose a target string into...
4    Compute nCk (binomial coefficient) - the numb...
5    '''\n    Perform a depth-first search travers...
6    Finds a subset of a list whose elements sum t...
7    Calculates the Levenshtein distance between t...
8    """Check if a graph is a directed acyclic gra...
9    """\nRecursively compute a value by passing n...
10   Compute the difference between sparse matrice...
11   Finds the length of the longest increasing an...
12   '''Iteratively find the median of a list of n...
13   """Find all the prime factors of a positive i...
14   Calculate the intersection of two graphs as a...
15   """Find all subsets of a list whose sum equal...
16   """Sums or multiplies string elements of an e...
17   ''' \nCheck whether a s

In [21]:
data_funcs.to_csv("docstring_generation_funcs.csv", index=False)

In [22]:
# Run the pipeline on the classes data
data_classes = generate_docstrings_pipeline(data_classes, models_classes)


Generating docstrings by Gemini-1.0-pro...


C:\Users\user\AppData\Local\Temp\ipykernel_29276\435095776.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data[new_docstring_col] = None


  0%|          | 0/10 [00:00<?, ?it/s]

                                       Gemini-1.0-pro
40  """A class used to represent a triangle.\n\nAt...
41  A class used to represent a worker and their d...
42  A class that represents a binary number and al...
43  """A class used to represent a 2D point.\n\nAt...
44  A class used to simulate a roulette game.\n\n ...
45  """A class used to represent an investment.\n\...
46  """A class used to represent a restaurant.\n\n...
47  A class used to represent a polynomial.\n\n   ...
48  A todo list class to manage and track tasks.\n...
49  A class used to represent a recipe book.\n\n  ...

Generating docstrings by GPT-3.5 Turbo...


C:\Users\user\AppData\Local\Temp\ipykernel_29276\435095776.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data[new_docstring_col] = None


  0%|          | 0/10 [00:00<?, ?it/s]

                                        GPT-3.5 Turbo
40  A class representing a triangle.\n\nAttributes...
41  \nA class representing a worker.\n\nAttributes...
42  \nA class used to perform binary arithmetic op...
43  A class representing a 2D point with x and y c...
44  A class representing a roulette game.\n\nAttri...
45  \nA class representing investments with method...
46  A class used to represent a restaurant.\n\nAtt...
47  \nA class used to represent a polynomial.\n\nA...
48  \n    A class representing a todo list.\n\n   ...
49  \nA class representing a recipe book.\n\nAttri...

Generating docstrings by Claude-instant-1...


C:\Users\user\AppData\Local\Temp\ipykernel_29276\435095776.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data[new_docstring_col] = None


  0%|          | 0/10 [00:00<?, ?it/s]

                                     Claude-instant-1
40   """A triangle class that stores sides and ang...
41   """A class used to represent a worker.\n\nAtt...
42   """A class to perform binary arithmetic opera...
43   """A point in 2D space.\n\nAttributes:\n-----...
44   """A Roulette simulation game. \n\nThis class...
45   """A class used to model investment accounts....
46   """A class used to represent a restaurant.\n\...
47   A class representing a polynomial with coeffi...
48   """A class used to manage tasks added to a to...
49   """A class used to store recipes.\n\n    Attr...


In [23]:
data_classes.to_csv("docstring_generation_class.csv", index=False)

In [24]:
# Concatenate the dataframes and save the final csv
data_full_docstrings_generated = pd.concat([data_funcs, data_classes], ignore_index=True)
data_full_docstrings_generated.to_csv("data_full_docstrings_generated.csv", index=False)